# Tutorial 03: Efficient Sampling with Tinker

Tinker runs on remote GPUs. Every API call involves network latency plus GPU compute time. If you send sampling requests one at a time -- send, wait, send, wait -- you spend most of your time idle while Tinker works.

The solution: send requests **concurrently** as futures. Tinker can batch and pipeline concurrent requests on the GPU, so N requests take far less than N times the cost of one request. This matters most for sampling, where RL training may require hundreds of completions per step.

In [1]:
import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import tinker  # noqa: E402

from tinker_cookbook.renderers import get_renderer, get_text_content  # noqa: E402

## Setup

We create a `SamplingClient` for the base Qwen3.5-4B model (no fine-tuning needed for this tutorial). We also set up a renderer to handle the chat template and a list of diverse prompts to sample from.

In [2]:
BASE_MODEL = "Qwen/Qwen3.5-4B"

service_client = tinker.ServiceClient()
sampling_client = service_client.create_sampling_client(base_model=BASE_MODEL)
tokenizer = sampling_client.get_tokenizer()
renderer = get_renderer("qwen3_5", tokenizer)

stop_sequences = renderer.get_stop_sequences()
params = tinker.SamplingParams(max_tokens=150, temperature=0.7, stop=stop_sequences)

# A diverse set of prompts to sample from
prompts = [
    "What causes thunder?",
    "Write a haiku about the ocean.",
    "What is the capital of New Zealand?",
    "Explain what a hash table is in two sentences.",
    "Name three inventions from the 19th century.",
    "Why do leaves change color in autumn?",
    "Translate to Spanish: The library closes at nine.",
    "What is the smallest prime number greater than 50?",
]

print(f"Model: {BASE_MODEL}")
print(f"Prompts: {len(prompts)}")

Model: Qwen/Qwen3.5-4B
Prompts: 8


## Sequential sampling (the slow way)

The simplest approach: for each prompt, build the generation input, call `sample()`, immediately call `.result()` to block until it finishes, then move on to the next. Each request waits for the previous one to complete before starting.

In [3]:
start = time.time()
sequential_results = []

for prompt_text in prompts:
    messages = [{"role": "user", "content": prompt_text}]
    model_input = renderer.build_generation_prompt(messages)

    # Block on each request before sending the next
    result = sampling_client.sample(
        prompt=model_input, num_samples=1, sampling_params=params
    ).result()
    response_msg, _ = renderer.parse_response(result.sequences[0].tokens)
    sequential_results.append(get_text_content(response_msg))

sequential_time = time.time() - start

for prompt_text, answer in zip(prompts, sequential_results):
    print(f"Q: {prompt_text}")
    print(f"A: {answer[:120]}...\n")

print(
    f"Sequential: {sequential_time:.1f}s for {len(prompts)} prompts ({sequential_time / len(prompts):.1f}s each)"
)

Q: What causes thunder?
A: Thinking Process:

1.  **Analyze the Request:**
    *   Question: "What causes thunder?"
    *   Intent: The user wants ...

Q: Write a haiku about the ocean.
A: Thinking Process:

1.  **Analyze the Request:**
    *   Topic: The ocean.
    *   Form: Haiku (5-7-5 syllables).

2.  **...

Q: What is the capital of New Zealand?
A: Thinking Process:

1.  **Identify the core question:** The user is asking for the capital of New Zealand.
2.  **Access k...

Q: Explain what a hash table is in two sentences.
A: Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Hash Table.
    *   Constraint: Exactly two sentences.
  ...

Q: Name three inventions from the 19th century.
A: Thinking Process:

1.  **Analyze the Request:**
    *   Task: Name three inventions.
    *   Constraint: They must be fr...

Q: Why do leaves change color in autumn?
A: Here's a thinking process that leads to the explanation of why leaves change color in autumn:

1.  **Analyze the Request.

## Concurrent sampling with futures

`sample()` returns a future immediately -- the request is already in flight before you call `.result()`. The key insight: submit **all** requests first, then collect results. Tinker batches concurrent requests on the GPU for higher throughput.

In [4]:
start = time.time()

# Step 1: Submit ALL requests (non-blocking -- each returns a future immediately)
futures = []
for prompt_text in prompts:
    messages = [{"role": "user", "content": prompt_text}]
    model_input = renderer.build_generation_prompt(messages)
    future = sampling_client.sample(prompt=model_input, num_samples=1, sampling_params=params)
    futures.append(future)

# Step 2: Collect results (all requests were running in parallel)
concurrent_results = []
for future in futures:
    result = future.result()
    response_msg, _ = renderer.parse_response(result.sequences[0].tokens)
    concurrent_results.append(get_text_content(response_msg))

concurrent_time = time.time() - start

for prompt_text, answer in zip(prompts, concurrent_results):
    print(f"Q: {prompt_text}")
    print(f"A: {answer[:120]}...\n")

print(f"Concurrent: {concurrent_time:.1f}s for {len(prompts)} prompts")
print(f"Sequential: {sequential_time:.1f}s for {len(prompts)} prompts")
print(f"Speedup: {sequential_time / concurrent_time:.1f}x")

Q: What causes thunder?
A: Here's a thinking process that leads to the explanation of thunder:

1.  **Analyze the Request:**
    *   **Question:** ...

Q: Write a haiku about the ocean.
A: Thinking Process:

1.  **Analyze the Request:**
    *   Topic: The ocean.
    *   Form: Haiku (5-7-5 syllables).

2.  **...

Q: What is the capital of New Zealand?
A: Thinking Process:

1.  **Identify the core question:** The user is asking for the capital of New Zealand.

2.  **Access ...

Q: Explain what a hash table is in two sentences.
A: Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Hash table.
    *   Constraint: Exactly two sentences.
  ...

Q: Name three inventions from the 19th century.
A: Thinking Process:

1.  **Analyze the Request:**
    *   Task: Name three inventions.
    *   Constraint: They must be fr...

Q: Why do leaves change color in autumn?
A: Here's a thinking process that leads to the explanation of why leaves change color in autumn:

1.  **Analyze the Request.

## Multiple completions per prompt (num_samples)

In GRPO-style RL, you need `group_size` independent completions for each problem so you can compare them and compute advantages. The `num_samples` parameter generates multiple completions in a single API call -- more efficient than sending separate requests for the same prompt.

In [5]:
GROUP_SIZE = 4
test_prompt = "Name a famous scientist and explain their key contribution in one sentence."

messages = [{"role": "user", "content": test_prompt}]
model_input = renderer.build_generation_prompt(messages)

# Single call with num_samples=4 -- generates 4 independent completions
start = time.time()
result = sampling_client.sample(
    prompt=model_input, num_samples=GROUP_SIZE, sampling_params=params
).result()
multi_time = time.time() - start

print(f"Prompt: {test_prompt}\n")
for i, seq in enumerate(result.sequences):
    response_msg, _ = renderer.parse_response(seq.tokens)
    text = get_text_content(response_msg)
    print(f"Completion {i + 1}: {text[:150]}\n")

# Compare: 4 sequential single calls
start = time.time()
for _ in range(GROUP_SIZE):
    sampling_client.sample(prompt=model_input, num_samples=1, sampling_params=params).result()
sequential_multi_time = time.time() - start

print(f"num_samples={GROUP_SIZE} in one call: {multi_time:.1f}s")
print(f"{GROUP_SIZE} sequential calls:        {sequential_multi_time:.1f}s")
print(f"Speedup: {sequential_multi_time / multi_time:.1f}x")

Prompt: Name a famous scientist and explain their key contribution in one sentence.

Completion 1: Thinking Process:

1.  **Analyze the Request:**
    *   Task: Name a famous scientist and explain their key contribution.
    *   Constraint: Explain 

Completion 2: Thinking Process:

1.  **Analyze the Request:**
    *   Task: Name a famous scientist and explain their key contribution.
    *   Constraint: The expl

Completion 3: Thinking Process:

1.  **Analyze the Request:**
    *   Task: Name a famous scientist and explain their key contribution.
    *   Constraint: The expl

Completion 4: Thinking Process:

1.  **Analyze the Request:**
    *   Task: Name a famous scientist and explain their key contribution.
    *   Constraint: Explain 



num_samples=4 in one call: 5.5s
4 sequential calls:        18.7s
Speedup: 3.4x


## Putting it together: batch evaluation

Combine both techniques -- concurrent futures across prompts and `num_samples` per prompt -- for maximum throughput. This is exactly the pattern used in RL training: submit many sampling requests in parallel, each generating a group of completions, then collect and grade them all.

In [6]:
GROUP_SIZE = 4

start = time.time()

# Submit all requests concurrently, each with num_samples=GROUP_SIZE
futures = []
for prompt_text in prompts:
    messages = [{"role": "user", "content": prompt_text}]
    model_input = renderer.build_generation_prompt(messages)
    future = sampling_client.sample(
        prompt=model_input, num_samples=GROUP_SIZE, sampling_params=params
    )
    futures.append((prompt_text, future))

# Collect all results
total_completions = 0
for prompt_text, future in futures:
    result = future.result()
    completions = []
    for seq in result.sequences:
        response_msg, _ = renderer.parse_response(seq.tokens)
        completions.append(get_text_content(response_msg))
    total_completions += len(completions)
    print(f"Q: {prompt_text}")
    print(f"   ({len(completions)} completions, showing first): {completions[0][:100]}...\n")

batch_time = time.time() - start

print(f"Total: {total_completions} completions in {batch_time:.1f}s")
print(f"Throughput: {total_completions / batch_time:.1f} completions/second")

Q: What causes thunder?
   (4 completions, showing first): Thinking Process:

1.  **Analyze the Request:**
    *   Question: "What causes thunder?"
    *   Int...



Q: Write a haiku about the ocean.
   (4 completions, showing first): Thinking Process:

1.  **Analyze the Request:**
    *   Topic: The ocean.
    *   Form: Haiku (5-7-5...

Q: What is the capital of New Zealand?
   (4 completions, showing first): <think>
Thinking Process:

1.  **Identify the core question:** The user is asking for the capital ci...

Q: Explain what a hash table is in two sentences.
   (4 completions, showing first): Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Hash table.
    *   Constraint: Expla...

Q: Name three inventions from the 19th century.
   (4 completions, showing first): Thinking Process:

1.  **Analyze the Request:**
    *   Task: Name three inventions.
    *   Constra...

Q: Why do leaves change color in autumn?
   (4 completions, showing first): Here's a thinking process that leads to the explanation of why leaves change color in autumn:

1.  *...

Q: Translate to Spanish: The library closes at nine.
   (4 completions, showing first): 

## Next steps

This tutorial showed the two key techniques for efficient sampling: **concurrent futures** (submit all requests before collecting results) and **num_samples** (generate multiple completions per call). Together, they give you high throughput with minimal code changes.

- **Tutorial 04** (`04_first_rl.ipynb`): Uses this exact pattern -- sample many completions, grade them with a reward function, and train with GRPO.
- **Async docs** (`docs/async.mdx`): Full reference for sync/async APIs, the double-await pattern, and overlapping training requests.